# Quality and alignment

Scope: Quality and alignment in the WESAD protocol.

Input: Official pickles or synthetic signals; QC thresholds in YAML.

Output and saved artifacts: `qc/signal_qc.csv`, `window_coverage.csv`; features; QC and PSD figures; `protocol_lock.json` (under the configured output root; real `.parquet`, synthetic `.csv`).

Dependencies: Environment from `requirements.txt`, `src/` modules; setup dependencies only in notebook 00.

Manual checks: Read `README.md`, `PROTOCOL.md`, and the corresponding row in `FILE_GUIDE.md`; do not interpret the smoke test as a WESAD result. No precompiled output.

With raw files only, the notebook produces/consults `artifacts/real_raw_audit/` and does not build unsynchronized merges.


In [1]:
from pathlib import Path
import os, sys
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'configs/config.yaml').is_file() and (p / 'src').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter inside the project folder; no machine-specific path is required')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.config import load, setup, smoke_config
from src import workflow as w
SYNTHETIC = os.environ.get('WESAD_SYNTHETIC', '0') == '1'
c = load()
if SYNTHETIC: c = smoke_config(c)

# Reuse saved protocol instead of silently resetting extraction settings.
if not SYNTHETIC:
    import copy, json
    from src.config import fingerprint
    protocol_file = w.root(c) / 'protocol_lock.json'
    if protocol_file.is_file():
        from src.provenance import verify_lock
        saved_protocol = verify_lock(protocol_file)
        frozen = saved_protocol['config']
        if fingerprint(frozen) != saved_protocol['config_sha256']:
            raise ValueError('Saved protocol hash mismatch')
        c = copy.deepcopy(frozen)

setup(c)
print('Data mode:', 'SYNTHETIC EXECUTION ONLY' if SYNTHETIC else 'REAL WESAD')


Data mode: REAL WESAD


In [ ]:
from pathlib import Path
from src.wesad_io import discover

if SYNTHETIC:
    from src.synthetic import make_records
    records = make_records(seed=c["splits"]["seed"])
else:
    # Cerca i dati nella cartella raw configurata, incluse le sottocartelle.
    raw_dir = Path(c["dataset"]["raw_dir"])
    if not raw_dir.is_absolute():
        raw_dir = ROOT / raw_dir

    c["dataset"]["raw_dir"] = str(raw_dir.resolve())

    # Impostare True per i file del dataset ufficiale WESAD.
    c["dataset"]["trusted_pickle"] = True

    subject_files = discover(c["dataset"]["raw_dir"])
    print(f"Trovati {len(subject_files)} soggetti in {raw_dir}")

    records = w.load_records(c)

print(f"Caricati {len(records)} soggetti")


## ACC Diagnostics: lag screening

Resampled ACC magnitude with anti-aliasing filter at 8 Hz; non-overlapping 300 s segments. Positive lag: wrist pattern following chest pattern. Fixed diagnostic thresholds: |lag| > 2 s, peak < 0.3, or on the ±15 s boundary. Differing movements between chest and wrist, periodicity, and weak peaks prevent this from being considered a definitive verification of synchronization. No automatic correction.


In [3]:
from src.sync_diagnostic import estimate
import pandas as pd
from IPython.display import display
sync_rows=[]
if SYNTHETIC:
    print('Diagnostica non interpretata fisiologicamente sui dati sintetici.')
else:
    for record in records: sync_rows.extend(estimate(record))
    sync_diagnostics=pd.DataFrame(sync_rows)
    sync_diagnostics.to_csv(w.root(c)/'qc/acc_lag_segments.csv',index=False)
    display(sync_diagnostics)
    if 'lag_seconds' in sync_diagnostics:
        sync_subjects=sync_diagnostics.groupby('subject_id').agg(
            median_lag_seconds=('lag_seconds','median'),max_abs_lag_seconds=('lag_seconds',lambda x:x.abs().max()),
            anomalous_segments=('anomalous_lag','sum'),weak_segments=('weak_peak','sum'),boundary_segments=('boundary_peak','sum'))
        sync_subjects.to_csv(w.root(c)/'qc/acc_lag_subjects.csv')
        display(sync_subjects)
        print('Soggetti da ispezionare:')
        display(sync_subjects[(sync_subjects.anomalous_segments>0)|(sync_subjects.weak_segments>0)|(sync_subjects.boundary_segments>0)])


,subject_id,start_seconds,lag_seconds,peak_correlation,status,anomalous_lag,weak_peak,boundary_peak
0,10,0.0,0.000,0.125985,screening_only,False,True,False
1,10,300.0,5.125,0.051666,screening_only,True,True,False
2,10,600.0,2.125,0.117882,screening_only,True,True,False
3,10,900.0,0.500,0.284503,screening_only,False,True,False
4,10,1200.0,-2.875,0.131817,screening_only,True,True,False
...,...,...,...,...,...,...,...,...
277,9,3600.0,-0.750,0.159392,screening_only,False,True,False
278,9,3900.0,3.500,0.372981,screening_only,True,False,False
279,9,4200.0,-13.375,0.202671,screening_only,True,True,False
280,9,4500.0,-13.625,-0.061151,screening_only,True,True,False


,median_lag_seconds,max_abs_lag_seconds,anomalous_segments,weak_segments,boundary_segments
subject_id,,,,,
2,-0.4375,10.500,4,20,0
3,0.7500,14.875,7,19,0
4,0.0000,15.000,6,21,1
5,0.0625,14.500,7,16,0
6,0.0000,8.750,7,23,0
7,0.8750,9.375,5,13,0
8,-0.3750,14.625,5,18,0
9,0.6250,15.000,7,14,1
10,0.1250,14.500,8,16,0


Soggetti da ispezionare:


,median_lag_seconds,max_abs_lag_seconds,anomalous_segments,weak_segments,boundary_segments
subject_id,,,,,
2,-0.4375,10.500,4,20,0
3,0.7500,14.875,7,19,0
4,0.0000,15.000,6,21,1
5,0.0625,14.500,7,16,0
6,0.0000,8.750,7,23,0
7,0.8750,9.375,5,13,0
8,-0.3750,14.625,5,18,0
9,0.6250,15.000,7,14,1
10,0.1250,14.500,8,16,0
